In [ ]:
import pandas as pd
import numpy as np
import re

In [2]:
# load the data
movie_col_names = ['movie_id', 'title', 'genre']
movies = pd.read_csv('./movies_database/movies.dat', sep="::", header=None, names=movie_col_names, engine="python", encoding="latin1")
movies.head()

,movie_id,title,genre
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
users_col_names = ['user_id', 'gender', 'age', 'occupation', 'zip']
users = pd.read_csv('./movies_database/users.dat', sep="::", header=None, names=users_col_names, engine="python")
users.head()

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [4]:
ratings_col_names = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('./movies_database/ratings.dat', sep="::", header=None, names=ratings_col_names, engine="python")
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [5]:
# we want to find the best rated movie

In [6]:
data = pd.merge(movies, ratings)
data.head()

,movie_id,title,genre,user_id,rating,timestamp
0,1,Toy Story (1995),Animation|Children's|Comedy,1,5,978824268
1,1,Toy Story (1995),Animation|Children's|Comedy,6,4,978237008
2,1,Toy Story (1995),Animation|Children's|Comedy,8,4,978233496
3,1,Toy Story (1995),Animation|Children's|Comedy,9,5,978225952
4,1,Toy Story (1995),Animation|Children's|Comedy,10,5,978226474


In [7]:
to_drop = ['timestamp', 'user_id']
data.drop(columns=to_drop, inplace=True)
data.head()

,movie_id,title,genre,rating
0,1,Toy Story (1995),Animation|Children's|Comedy,5
1,1,Toy Story (1995),Animation|Children's|Comedy,4
2,1,Toy Story (1995),Animation|Children's|Comedy,4
3,1,Toy Story (1995),Animation|Children's|Comedy,5
4,1,Toy Story (1995),Animation|Children's|Comedy,5


In [8]:
years = []
title = [] 
for values in data['title']:
    years.append(int(re.search(r'(\([1-3][0-9]{3})\)', values).group()[1:5]))
    title.append(re.sub(r'(\([1-3][0-9]{3})\)', '', values))

data['year'] = years
data['title'] = title
data.head()

,movie_id,title,genre,rating,year
0,1,Toy Story,Animation|Children's|Comedy,5,1995
1,1,Toy Story,Animation|Children's|Comedy,4,1995
2,1,Toy Story,Animation|Children's|Comedy,4,1995
3,1,Toy Story,Animation|Children's|Comedy,5,1995
4,1,Toy Story,Animation|Children's|Comedy,5,1995


In [9]:
genres = data['genre'].str.split('|')

g = set([])
for i in genres:
    g = g.union(set(i))
    
g

{'Action',
 'Adventure',
 'Animation',
 "Children's",
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western'}

In [10]:
# data addition
data_extended = data

# slow as hell
# for index, row in data_extended.iterrows():
#    for j in g:
#        if j in row['genre']:
#            data_extended.loc[index, j] = True

# vectorization is required 
for j in g:
    data_extended[j] = data_extended['genre'].str.contains(str(j))

In [11]:
data_extended.drop(columns=['genre'], inplace=True)
data_extended.head()

,movie_id,title,rating,year,Comedy,Action,Documentary,Drama,Sci-Fi,Western,...,Animation,Mystery,Children's,Horror,Musical,War,Crime,Film-Noir,Fantasy,Romance
0,1,Toy Story,5,1995,True,False,False,False,False,False,...,True,False,True,False,False,False,False,False,False,False
1,1,Toy Story,4,1995,True,False,False,False,False,False,...,True,False,True,False,False,False,False,False,False,False
2,1,Toy Story,4,1995,True,False,False,False,False,False,...,True,False,True,False,False,False,False,False,False,False
3,1,Toy Story,5,1995,True,False,False,False,False,False,...,True,False,True,False,False,False,False,False,False,False
4,1,Toy Story,5,1995,True,False,False,False,False,False,...,True,False,True,False,False,False,False,False,False,False


In [12]:
data_extended.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 22 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   movie_id     1000209 non-null  int64
 1   title        1000209 non-null  str  
 2   rating       1000209 non-null  int64
 3   year         1000209 non-null  int64
 4   Comedy       1000209 non-null  bool 
 5   Action       1000209 non-null  bool 
 6   Documentary  1000209 non-null  bool 
 7   Drama        1000209 non-null  bool 
 8   Sci-Fi       1000209 non-null  bool 
 9   Western      1000209 non-null  bool 
 10  Thriller     1000209 non-null  bool 
 11  Adventure    1000209 non-null  bool 
 12  Animation    1000209 non-null  bool 
 13  Mystery      1000209 non-null  bool 
 14  Children's   1000209 non-null  bool 
 15  Horror       1000209 non-null  bool 
 16  Musical      1000209 non-null  bool 
 17  War          1000209 non-null  bool 
 18  Crime        1000209 non-null  bool 
 19  Film-Noir  

In [13]:
data_extended.describe()

,movie_id,rating,year
count,1.000209e+06,1.000209e+06,1.000209e+06
mean,1.865540e+03,3.581564e+00,1.986698e+03
std,1.096041e+03,1.117102e+00,1.434933e+01
min,1.000000e+00,1.000000e+00,1.919000e+03
25%,1.030000e+03,3.000000e+00,1.982000e+03
50%,1.835000e+03,4.000000e+00,1.992000e+03
75%,2.770000e+03,4.000000e+00,1.997000e+03
max,3.952000e+03,5.000000e+00,2.000000e+03


In [14]:
# pivot table
mean_ratings = data.pivot_table(values='rating', index='title', aggfunc=[np.min, np.max, np.mean, len])
mean_ratings

,min,max,mean,len
,rating,rating,rating,rating
title,,,,
"$1,000,000 Duck",1,5,3.027027,37
'Night Mother,1,5,3.371429,70
'Til There Was You,1,5,2.692308,52
"'burbs, The",1,5,2.910891,303
...And Justice for All,1,5,3.713568,199
...,...,...,...,...
"Zed & Two Noughts, A",1,5,3.413793,29
Zero Effect,1,5,3.750831,301


In [15]:
# basic information about data
mean_ratings.describe()

,min,max,mean,len
,rating,rating,rating,rating
count,3664.000000,3664.000000,3664.000000,3664.000000
mean,1.177948,4.794487,3.237347,272.982806
std,0.554922,0.627047,0.674236,388.065233
min,1.000000,1.000000,1.000000,1.000000
25%,1.000000,5.000000,2.820216,33.000000
50%,1.000000,5.000000,3.329545,123.500000
75%,1.000000,5.000000,3.740150,354.000000
max,5.000000,5.000000,5.000000,3428.000000


In [16]:
# this is where we ended last time
ratings_by_title = data.groupby('title').size() >= 250
mean_ratings.loc[ratings_by_title].sort_values(by=('len', 'rating'), ascending=False).head(10)

,min,max,mean,len
,rating,rating,rating,rating
title,,,,
American Beauty,1,5,4.317386,3428
Star Wars: Episode IV - A New Hope,1,5,4.453694,2991
Star Wars: Episode V - The Empire Strikes Back,1,5,4.292977,2990
Star Wars: Episode VI - Return of the Jedi,1,5,4.022893,2883
Jurassic Park,1,5,3.763847,2672
Saving Private Ryan,1,5,4.337354,2653
Terminator 2: Judgment Day,1,5,4.058513,2649
"Matrix, The",1,5,4.315830,2590


In [17]:
# domain knowledge tells us that the result is not correct
# think and do a better analysis (top rated movie)
# hint: one of the top rated movies should be "Shawshank Redemption, The"

In [18]:
# celkový průměr
C = data['rating'].mean() #průměrné hodnocení všech filmů
m = 250  # dolní hranice pro počet hlasů
mean_ratings[('bayesian_avg', 'rating')] = (mean_ratings[('mean', 'rating')] * mean_ratings[('len', 'rating')] + m * C) / (mean_ratings[('len', 'rating')] + m)
mean_ratings

,min,max,mean,len,bayesian_avg
,rating,rating,rating,rating,rating
title,,,,,
"$1,000,000 Duck",1,5,3.027027,37,3.510074
'Night Mother,1,5,3.371429,70,3.535597
'Til There Was You,1,5,2.692308,52,3.428447
"'burbs, The",1,5,2.910891,303,3.214089
...And Justice for All,1,5,3.713568,199,3.640069
...,...,...,...,...,...
"Zed & Two Noughts, A",1,5,3.413793,29,3.564126
Zero Effect,1,5,3.750831,301,3.674031


In [19]:
mean_ratings_sorted = mean_ratings.sort_values(by=('bayesian_avg', 'rating'), ascending=False)
mean_ratings_sorted

,min,max,mean,len,bayesian_avg
,rating,rating,rating,rating,rating
title,,,,,
"Shawshank Redemption, The",1,5,4.554558,2227,4.456355
"Godfather, The",1,5,4.524966,2223,4.429596
Schindler's List,1,5,4.510417,2304,4.419495
"Usual Suspects, The",1,5,4.517106,1783,4.402062
Raiders of the Lost Ark,1,5,4.477725,2514,4.396668
...,...,...,...,...,...
Superman IV: The Quest for Peace,1,5,1.888554,332,2.615792
Super Mario Bros.,1,5,1.874286,350,2.585652
